# PancrAI — Colab Server v2.0
### SwinUNETR v11.0 · Flask · ngrok public URL

**Setup:** Runtime → Change runtime type → **T4 GPU** → Save

Run cells **1 → 2 → 3** in order. Cell 3 prints your public URL.


In [1]:
# ================================================================
# CELL 1 — INSTALL DEPENDENCIES
# ================================================================
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

print("📦 Installing MONAI + medical imaging stack...")
pip('monai[nibabel,skimage,tqdm]>=1.3.0,<1.4.0', 'einops', 'timm')

print("📦 Installing Flask + web stack...")
pip('flask', 'flask-cors', 'pillow', 'numpy', 'scipy')

print("📦 Installing reportlab (PDF generation)...")
pip('reportlab')

print("📦 Installing pyngrok...")
pip('pyngrok==7.2.3')

import torch, monai, flask
print()
print(f"✅ PyTorch  : {torch.__version__}")
cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else 'CPU only'
print(f"✅ CUDA     : {cuda_ok} — {gpu_name}")
print(f"✅ MONAI    : {monai.__version__}")
print(f"✅ Flask    : {flask.__version__}")

if not cuda_ok:
    print()
    print("⚠️  WARNING: No GPU detected!")
    print("   Runtime → Change runtime type → T4 GPU → Save")
else:
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM     : {vram:.1f} GB")
    print()
    print("All dependencies ready — run Cell 2.")


📦 Installing MONAI + medical imaging stack...
📦 Installing Flask + web stack...
📦 Installing reportlab (PDF generation)...
📦 Installing pyngrok...

✅ PyTorch  : 2.11.0+cu128
✅ CUDA     : True — Tesla T4
✅ MONAI    : 1.3.2
✅ Flask    : 3.1.3
✅ VRAM     : 15.6 GB

All dependencies ready — run Cell 2.


/tmp/ipykernel_1888/3170396406.py:28: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"✅ Flask    : {flask.__version__}")


In [ ]:
# ================================================================
# CELL 2 — SYNC FILES FROM GOOGLE DRIVE
#
# Expected folder in My Drive:   deploy_v90/
# Required files:
#   best_model.pth   config.json   infer.py   app.py
# Templates folder:

#   templates/index.html   templates/viewer.html
# ================================================================
import os, shutil
from google.colab import drive

SERVER_DIR = '/content/pancrai_server'
TMPL_DIR   = f'{SERVER_DIR}/templates'
UPL_DIR    = f'{SERVER_DIR}/uploads'
OUT_DIR    = f'{SERVER_DIR}/outputs'

for d in [SERVER_DIR, TMPL_DIR, UPL_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('=' * 55)
print('  PancrAI v2.0 — Google Drive Sync')
print('=' * 55)

drive.mount('/content/drive')

# ── SET YOUR DRIVE FOLDER NAME HERE ──────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/Final_Deploy'
# ─────────────────────────────────────────────────────────────

print(f'\nReading from: {DRIVE_FOLDER}\n')

# Core files → SERVER_DIR
for fname in ['best_model.pth', 'config.json', 'infer.py', 'app.py']:
    src = f'{DRIVE_FOLDER}/{fname}'
    dst = f'{SERVER_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        mb = os.path.getsize(dst) / 1e6
        print(f'  ✅ {fname:<22}  ({mb:.1f} MB)')
    else:
        print(f'  ❌ Not found: {src}')

# HTML templates → templates/
for fname in ['index.html', 'viewer.html']:
    src = f'{DRIVE_FOLDER}/templates/{fname}'
    dst = f'{TMPL_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  ✅ templates/{fname}')
    else:
        print(f'  ❌ Not found: {src}')

# ── Verify ───────────────────────────────────────────────────
print('\n' + '─' * 50)
print('VERIFICATION:')
required = {
    f'{SERVER_DIR}/best_model.pth' : 'Model weights',
    f'{SERVER_DIR}/config.json'    : 'Config',
    f'{SERVER_DIR}/infer.py'       : 'Inference script',
    f'{SERVER_DIR}/app.py'         : 'Flask app',
    f'{TMPL_DIR}/index.html'       : 'Upload page',
    f'{TMPL_DIR}/viewer.html'      : 'Viewer page',
}

all_ok = True
for path, label in required.items():
    ok   = os.path.exists(path)
    size = f'({os.path.getsize(path)/1e6:.1f} MB)' if ok else ''
    print(f'  {"✅" if ok else "❌ MISSING"}  {label:<22}  {size}')
    if not ok: all_ok = False

print()
if all_ok:
    print('🎉 All files present — ready for Cell 3.')
else:
    print('⚠️  Fix missing files in your Drive folder, then re-run this cell.')


  PancrAI v2.0 — Google Drive Sync
Mounted at /content/drive

Reading from: /content/drive/MyDrive/Final_Deploy

  ✅ best_model.pth          (256.3 MB)
  ✅ config.json             (0.0 MB)
  ✅ infer.py                (0.0 MB)
  ✅ app.py                  (0.1 MB)
  ✅ templates/index.html
  ✅ templates/viewer.html

──────────────────────────────────────────────────
VERIFICATION:
  ✅  Model weights           (256.3 MB)
  ✅  Config                  (0.0 MB)
  ✅  Inference script        (0.0 MB)
  ✅  Flask app               (0.1 MB)
  ✅  Upload page             (0.0 MB)
  ✅  Viewer page             (0.0 MB)

🎉 All files present — ready for Cell 3.


In [ ]:
# ================================================================
# CELL 3 — START FLASK + NGROK
#
# 1. Get token: https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Paste below and run
# ================================================================
import os, sys, threading, time, subprocess
from pyngrok import ngrok, conf

# ── YOUR NGROK TOKEN ──────────────────────────────────────────
NGROK_TOKEN = '3CpoKJTYeUKDCeu3S7CZuDd3vFO_54ZvmotKr3FZD3BUroVbk'   # ← edit this
# ─────────────────────────────────────────────────────────────

if NGROK_TOKEN == 'PASTE_YOUR_TOKEN_HERE':
    raise ValueError(
        "Set your ngrok token!\n"
        "  1. https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "  2. Replace PASTE_YOUR_TOKEN_HERE above"
    )

SERVER_DIR = '/content/pancrai_server'
PORT       = 5020

# ── Kill stale processes ──────────────────────────────────────
subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
time.sleep(1)
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass
ngrok.kill()
time.sleep(1)

# ── Start Flask in background ─────────────────────────────────
import importlib.util

flask_ready  = threading.Event()
flask_errors = []

def run_flask():
    if SERVER_DIR not in sys.path:
        sys.path.insert(0, SERVER_DIR)
    os.environ['PANCRAI_MODEL']   = f'{SERVER_DIR}/best_model.pth'
    os.environ['PANCRAI_CONFIG']  = f'{SERVER_DIR}/config.json'
    os.environ['PANCRAI_UPLOADS'] = f'{SERVER_DIR}/uploads'
    os.environ['PANCRAI_OUTPUTS'] = f'{SERVER_DIR}/outputs'

    spec = importlib.util.spec_from_file_location('pancrai_app', f'{SERVER_DIR}/app.py')
    mod  = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(mod)
        flask_ready.set()
        mod.app.run(debug=False, host='0.0.0.0', port=PORT,
                    threaded=True, use_reloader=False)
    except Exception as e:
        import traceback
        flask_errors.append(traceback.format_exc())
        flask_ready.set()

t = threading.Thread(target=run_flask, daemon=True)
t.start()

print('⏳ Starting Flask...', end='', flush=True)
flask_ready.wait(timeout=25)
time.sleep(2)

if flask_errors:
    print(' FAILED\n')
    print('❌ Error:\n' + flask_errors[0])
    raise RuntimeError('Flask failed to start.')
print(' ready ✅')

# ── Start ngrok tunnel ────────────────────────────────────────
conf.get_default().auth_token = NGROK_TOKEN
tunnel     = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url

print()
print('╔══════════════════════════════════════════════════════╗')
print(f'║  🌐  PancrAI is LIVE at:                             ')
print(f'║  👉  {public_url:<50}  ║')
print('╚══════════════════════════════════════════════════════╝')
print()
print(f'  Open the link above in any browser.')
print(f'  Share with your supervisor / examiner directly.')
print(f'  Model: {"✅ loaded" if __import__("os").path.exists(f"{SERVER_DIR}/best_model.pth") else "⚠️  demo mode"}')


⏳ Starting Flask... * Serving Flask app 'pancrai_app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5020
 * Running on http://172.28.0.12:5020
INFO:werkzeug:Press CTRL+C to quit


 ready ✅


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://virtuous-backyard-blinking.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:
# ================================================================
# CELL 4 — SMOKE TEST  (optional)
# Run after Cell 3 to verify all endpoints respond.
# ================================================================
import requests

BASE = public_url   # set by Cell 3

def check(method, path, **kwargs):
    try:
        r = getattr(requests, method)(f'{BASE}{path}', timeout=15, **kwargs)
        ok = r.status_code < 400
        print(f'  {"✅" if ok else "⚠️ HTTP "+str(r.status_code)}  {method.upper()} {path}')
        return r
    except Exception as e:
        print(f'  ❌  {method.upper()} {path}  →  {e}')
        return None

print('Testing API endpoints...')
print()
r1 = check('get', '/')
r2 = check('get', '/status/nonexistent')
print()
if r1 and r1.status_code == 200:
    print(f'✅ Server healthy at {BASE}')
    print(f'   Open → {BASE}')


In [ ]:
# ================================================================
# CELL 5 — JOB MONITOR  (optional)
# Run at any time to see all inference jobs.
# ================================================================
import sys
SERVER_DIR = '/content/pancrai_server'
if SERVER_DIR not in sys.path:
    sys.path.insert(0, SERVER_DIR)

try:
    import pancrai_app as _app
    jobs = _app.jobs
    if not jobs:
        print('No jobs yet. Upload a CT scan via the web interface.')
    else:
        print(f'{"JOB ID":<14} {"STATUS":<10} {"PROG":>5}  {"MESSAGE"}')
        print('─' * 70)
        for jid, j in jobs.items():
            print(f"{jid:<14} {j['status']:<10} {str(j['progress'])+'%':>5}  {j['message'][:45]}")
except ImportError:
    print('Flask app not loaded yet. Run Cell 3 first.')


In [ ]:
# ================================================================
# CELL 6 — KEEP-ALIVE PING  (optional)
# Prevents Colab from disconnecting during long inference.
# Stop with the ■ button when done.
# ================================================================
import time, requests

BASE       = public_url
INTERVAL_S = 55
MAX_H      = 11

start = time.time()
pings = 0

print(f'Keep-alive: pinging {BASE}/ every {INTERVAL_S}s')
print('Stop with ■ (interrupt) when done.')
print()

try:
    while (time.time() - start) < MAX_H * 3600:
        try:
            r = requests.get(f'{BASE}/', timeout=10)
            pings += 1
            mins = (time.time() - start) / 60
            print(f'  [{mins:6.1f}m]  ping #{pings}  →  HTTP {r.status_code}', flush=True)
        except Exception as e:
            print(f'  Ping failed: {e}', flush=True)
        time.sleep(INTERVAL_S)
except KeyboardInterrupt:
    print(f'\nStopped after {pings} pings ({(time.time()-start)/60:.1f} min).')
